# Layer 4 — Model Predictor dan Benchmarking

Notebook ini melatih lima algoritma pada fitur Layer 3 dan membandingkan performanya.

Protokol evaluasi dijaga seperti ini:

1. Matriks **asli** (belum SMOTE) dibagi 80% latih dan 20% uji, `random_state=42`, stratifikasi pada `y`.
2. SMOTE hanya dijalankan pada data latih.
3. Data uji tetap berisi pengamatan sungguhan, dengan proporsi kelas yang sama seperti data sumber.

Berkas `layer3_smoted_features.csv` tetap dimuat agar bentuknya tercatat. Berkas itu sudah diseimbangkan sebelum ada pembagian latih-uji, sehingga tidak dipakai sebagai data uji.

In [1]:
# ==========================================
# ENVIRONMENT CONFIGURATION
# Ubah menjadi True jika dijalankan di Server Kampus (RAM/CPU besar)
# Ubah menjadi False jika dijalankan di Laptop Lokal
# ==========================================
RUN_ON_SERVER = False

from pathlib import Path
import gc
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier

warnings.filterwarnings(
    "ignore",
    message="`BaseEstimator._validate_data` is deprecated",
    category=FutureWarning,
)
sns.set_theme(style="whitegrid")

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "orders.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Folder proyek tidak ditemukan. "
        "Buka Jupyter dari folder Cross Selling Retail."
    )

PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "dataset"
RULES_PATH = PROJECT_DIR / "outputs" / "apriori_rules.csv"
SMOTE_PATH = PROJECT_DIR / "outputs" / "layer3_smoted_features.csv"
COMPARISON_PATH = PROJECT_DIR / "outputs" / "layer4_model_comparison.csv"
MODEL_PATH = PROJECT_DIR / "models" / "xgboost_cross_sell_model.pkl"
CHART_PATH = PROJECT_DIR / "outputs" / "layer4_model_comparison.png"
CM_PATH = PROJECT_DIR / "outputs" / "layer4_xgboost_confusion_matrix.png"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
COMPARISON_PATH.parent.mkdir(parents=True, exist_ok=True)

# SVC dengan probability=True melatih model utama lalu kalibrasi peluang.
# Kompleksitasnya kuadratik, jadi kelima model memakai cuplikan latih yang sama.
MAX_TRAIN_ROWS = 12_000
RANDOM_STATE = 42
N_MINING_ORDERS = 50_000
MAX_USERS = 12_000
CHUNK_SIZE = 1_000_000

if RUN_ON_SERVER:
    XGB_PARAMS = {
        "n_estimators": 1000,
        "max_depth": 8,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "tree_method": "hist",
    }
    print("--> [INFO] Berjalan dalam mode SERVER (Parameter Maksimal)")
else:
    XGB_PARAMS = {
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "tree_method": "auto",
    }
    print("--> [INFO] Berjalan dalam mode LAPTOP (Parameter Terbatas)")

print("--> [INFO] Parameter Layer 4 siap. SMOTE hanya akan diterapkan pada data latih.")
print(f"--> [INFO] Parameter XGBoost: {XGB_PARAMS}")
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")

FEATURE_COLUMNS = [
    "total_prior_orders",
    "avg_basket_size",
    "avg_days_between",
    "antecedent_rate",
    "rule_confidence",
    "rule_lift",
    "interest_confidence",
    "interest_lift",
]

## 1. Muat berkas SMOTE dan susun ulang data asli

Delapan kolom pertama adalah fitur `X`. Kolom `y` adalah target. Data asli disusun ulang dengan parameter Layer 3 yang sama, termasuk `random_state=42` pada pemilihan 12.000 pengguna.

In [2]:
# Catatan bentuk berkas Layer 3. Isi sintetisnya tidak masuk ke data uji.
print("--> [INFO] Memuat layer3_smoted_features.csv, lalu menyusun ulang matriks asli sebelum SMOTE...")
smoted_df = pd.read_csv(SMOTE_PATH)
print("layer3_smoted_features.csv")
print(smoted_df.shape)
del smoted_df
gc.collect()

rules = pd.read_csv(RULES_PATH)
products = pd.read_csv(DATA_DIR / "products.csv", usecols=["product_id", "product_name"])
products["product_name"] = (
    products["product_name"].str.replace("\xa0", " ", regex=False).str.strip()
)
name_to_id = dict(zip(products["product_name"], products["product_id"].astype(np.int32)))
item_names = sorted(set(rules["antecedents"]) | set(rules["consequents"]))
id_to_name = {int(name_to_id[name]): name for name in item_names}
relevant_ids = set(id_to_name)

orders = pd.read_csv(
    DATA_DIR / "orders.csv",
    usecols=["order_id", "user_id", "eval_set", "order_number", "days_since_prior_order"],
    dtype={
        "order_id": np.int32,
        "user_id": np.int32,
        "order_number": np.int16,
        "days_since_prior_order": np.float32,
    },
)
print("orders")
print(orders.shape)

mining_order_ids = set(
    orders.loc[orders["eval_set"].eq("prior"), "order_id"].head(N_MINING_ORDERS).tolist()
)
mining_users = set(orders.loc[orders["order_id"].isin(mining_order_ids), "user_id"].tolist())
train_users = set(orders.loc[orders["eval_set"].eq("train"), "user_id"].tolist())
eligible_users = np.array(sorted(train_users - mining_users), dtype=np.int32)
rng = np.random.default_rng(RANDOM_STATE)
sampled_users = np.sort(rng.choice(eligible_users, size=MAX_USERS, replace=False))
user_set = set(sampled_users.tolist())

prior_orders = orders.loc[
    orders["eval_set"].eq("prior") & orders["user_id"].isin(user_set),
    ["order_id", "user_id", "order_number", "days_since_prior_order"],
]
user_features = prior_orders.groupby("user_id", sort=False).agg(
    total_prior_orders=("order_number", "max"),
    avg_days_between=("days_since_prior_order", "mean"),
)
print("fitur perilaku")
print(user_features.shape)

order_to_user = dict(
    zip(prior_orders["order_id"].tolist(), prior_orders["user_id"].tolist())
)
selected_order_ids = set(order_to_user)
basket_counts = {}
history_parts = []
for chunk in pd.read_csv(
    DATA_DIR / "order_products__prior.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": np.int32, "product_id": np.int32},
    chunksize=CHUNK_SIZE,
):
    sub = chunk.loc[chunk["order_id"].isin(selected_order_ids)]
    if sub.empty:
        continue
    for order_id, n_items in sub.groupby("order_id", sort=False).size().items():
        basket_counts[int(order_id)] = basket_counts.get(int(order_id), 0) + int(n_items)
    relevant = sub.loc[sub["product_id"].isin(relevant_ids), ["order_id", "product_id"]]
    if not relevant.empty:
        history_parts.append(relevant)

avg_basket = (
    pd.Series(basket_counts, name="basket_size")
    .rename_axis("order_id")
    .reset_index()
    .assign(user_id=lambda frame: frame["order_id"].map(order_to_user))
    .groupby("user_id")["basket_size"]
    .mean()
)
user_features["avg_basket_size"] = avg_basket

product_events = pd.concat(history_parts, ignore_index=True)
product_events["user_id"] = product_events["order_id"].map(order_to_user).astype(np.int32)
product_events["product_name"] = product_events["product_id"].map(id_to_name)
product_history = (
    product_events.drop_duplicates(["order_id", "product_name"])
    .groupby(["user_id", "product_name"], sort=False)["order_id"]
    .nunique()
    .rename("n_orders")
    .reset_index()
)
print("riwayat produk aturan")
print(product_history.shape)

train_orders = orders.loc[
    orders["eval_set"].eq("train") & orders["user_id"].isin(user_set),
    ["order_id", "user_id"],
]
train_items = pd.read_csv(
    DATA_DIR / "order_products__train.csv",
    usecols=["order_id", "product_id"],
    dtype={"order_id": np.int32, "product_id": np.int32},
)
train_items = train_items.loc[
    train_items["order_id"].isin(set(train_orders["order_id"].tolist()))
    & train_items["product_id"].isin(relevant_ids),
    ["order_id", "product_id"],
]
train_items = train_items.merge(train_orders, on="order_id", how="inner")
train_items["product_name"] = train_items["product_id"].map(id_to_name)

users = user_features.index.to_numpy()
n_users = len(users)
n_rules = len(rules)
purchase_counts = (
    product_history.pivot(index="user_id", columns="product_name", values="n_orders")
    .reindex(index=users, columns=item_names)
    .fillna(0)
    .to_numpy(dtype=np.float32)
)
train_flags = (
    train_items.assign(bought=np.int8(1))
    .pivot_table(index="user_id", columns="product_name", values="bought", aggfunc="max", fill_value=0)
    .reindex(index=users, columns=item_names, fill_value=0)
    .to_numpy(dtype=np.int8)
)
name_to_col = {name: col for col, name in enumerate(item_names)}
antecedent_idx = np.array([name_to_col[name] for name in rules["antecedents"]])
consequent_idx = np.array([name_to_col[name] for name in rules["consequents"]])
total_orders = user_features.loc[users, "total_prior_orders"].to_numpy(np.float32)
antecedent_rate = purchase_counts[:, antecedent_idx] / total_orders[:, None]
rule_confidence = rules["confidence"].to_numpy(np.float32)
rule_lift = rules["lift"].to_numpy(np.float32)
target = train_flags[:, consequent_idx]

feature_df = pd.DataFrame(
    {
        "total_prior_orders": np.repeat(total_orders, n_rules),
        "avg_basket_size": np.repeat(user_features.loc[users, "avg_basket_size"].to_numpy(np.float32), n_rules),
        "avg_days_between": np.repeat(user_features.loc[users, "avg_days_between"].to_numpy(np.float32), n_rules),
        "antecedent_rate": antecedent_rate.reshape(-1),
        "rule_confidence": np.tile(rule_confidence, n_users),
        "rule_lift": np.tile(rule_lift, n_users),
        "interest_confidence": (antecedent_rate * rule_confidence).reshape(-1),
        "interest_lift": (antecedent_rate * rule_lift).reshape(-1),
        "y": target.reshape(-1),
    }
)
feature_df[FEATURE_COLUMNS] = feature_df[FEATURE_COLUMNS].astype(np.float32)
feature_df["y"] = feature_df["y"].astype(np.int8)
print("matriks asli sebelum SMOTE")
print(feature_df.shape)

del orders, prior_orders, product_history, train_orders, train_items, user_features
gc.collect()

layer3_smoted_features.csv
(519142, 9)
orders
(3421083, 5)
fitur perilaku
(12000, 2)
riwayat produk aturan
(29966, 3)
matriks asli sebelum SMOTE
(288000, 9)


0

## 2. Split 80:20, lalu SMOTE hanya pada data latih

Stratifikasi menjaga proporsi kelas 1 di data latih dan data uji. Setelah SMOTE, data latih diseimbangkan. Cuplikan bersama sebesar 12.000 baris lalu dipakai oleh kelima algoritma, termasuk SVM.

In [3]:
print("--> [INFO] Membagi data 80:20, lalu menyeimbangkan hanya data latih dengan SMOTE...")
X = feature_df[FEATURE_COLUMNS]
y = feature_df["y"].astype(np.int32)
print("X")
print(X.shape)
print("y")
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)
print("X_train sebelum SMOTE")
print(X_train.shape)
print("X_test")
print(X_test.shape)
print("proporsi kelas 1 pada data uji:", round(float(y_test.mean()), 4))

smote = SMOTE(sampling_strategy="auto", random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_sm = pd.DataFrame(X_train_sm, columns=FEATURE_COLUMNS).astype(np.float32)
y_train_sm = pd.Series(y_train_sm, name="y").astype(np.int32)
print("X_train setelah SMOTE")
print(X_train_sm.shape)
print(y_train_sm.value_counts().sort_index().to_string())

if len(X_train_sm) > MAX_TRAIN_ROWS:
    X_fit, _, y_fit, _ = train_test_split(
        X_train_sm,
        y_train_sm,
        train_size=MAX_TRAIN_ROWS,
        random_state=RANDOM_STATE,
        stratify=y_train_sm,
    )
else:
    X_fit, y_fit = X_train_sm, y_train_sm

print("X_fit yang dipakai kelima model")
print(X_fit.shape)
print(y_fit.value_counts().sort_index().to_string())

X
(288000, 8)
y
(288000,)
X_train sebelum SMOTE
(230400, 8)
X_test
(57600, 8)
proporsi kelas 1 pada data uji: 0.0987
X_train setelah SMOTE
(415314, 8)
y
0    207657
1    207657
X_fit yang dipakai kelima model
(12000, 8)
y
0    6000
1    6000


## 3. Lima model

XGBoost adalah model yang akan disimpan untuk integrasi. Hyperparameter-nya diambil dari dictionary `XGB_PARAMS` di sel konfigurasi, sesuai flag `RUN_ON_SERVER`. Empat model lain menjadi pembanding pada data latih yang sama.

SVM dan Regresi Logistik peka terhadap skala angka, jadi keduanya memakai `StandardScaler` yang dihitung dari data latih saja. Pohon keputusan tidak membutuhkan skala itu.

In [4]:
print("--> [INFO] Menyiapkan 5 model: XGBoost, LightGBM, Random Forest, SVM, dan Logistic Regression...")
models = {
    "XGBoost": XGBClassifier(
        **XGB_PARAMS,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=200,
        learning_rate=0.1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "SVM": Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                SVC(
                    kernel="rbf",
                    probability=True,
                    random_state=RANDOM_STATE,
                    cache_size=1000,
                ),
            ),
        ]
    ),
    "Logistic Regression": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
}
print("jumlah model:", len(models))

jumlah model: 5


## 4. Pelatihan dan metrik pada data uji

Precision, recall, dan F1 memakai rata-rata macro agar kelas 0 dan kelas 1 berbobot sama. AUC-ROC memakai peluang kelas 1. Akurasi global tetap dilaporkan, tetapi pada data uji yang tidak seimbang angka ini mudah terlihat tinggi.

In [5]:
print("--> [INFO] Melatih kelima model pada data latih yang sama dan mengevaluasi data uji...")
def evaluate_model(name, model, X_test_data, y_test_data):
    # Peluang kelas 1 diperlukan untuk AUC-ROC, bukan hanya label 0/1.
    pred = model.predict(X_test_data)
    proba = model.predict_proba(X_test_data)[:, 1]
    return pred, {
        "Model": name,
        "Akurasi": accuracy_score(y_test_data, pred),
        "Precision Macro": precision_score(y_test_data, pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_test_data, pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_test_data, pred, average="macro", zero_division=0),
        "AUC-ROC": roc_auc_score(y_test_data, proba),
    }


fitted_models = {}
predictions = {}
rows = []
for name, model in models.items():
    started = time.perf_counter()
    model.fit(X_fit, y_fit)
    pred, metrics = evaluate_model(name, model, X_test, y_test)
    fitted_models[name] = model
    predictions[name] = pred
    rows.append(metrics)
    print(f"{name} selesai dalam {time.perf_counter() - started:.1f} detik")

comparison = (
    pd.DataFrame(rows)
    .sort_values(["F1 Macro", "AUC-ROC"], ascending=False)
    .reset_index(drop=True)
)
comparison.to_csv(COMPARISON_PATH, index=False)
print("tabel komparasi")
print(comparison.shape)
print(comparison.round(4).to_string(index=False))

XGBoost selesai dalam 0.6 detik
LightGBM selesai dalam 1.8 detik
Random Forest selesai dalam 1.3 detik
SVM selesai dalam 44.9 detik
Logistic Regression selesai dalam 0.2 detik
tabel komparasi
(5, 6)
              Model  Akurasi  Precision Macro  Recall Macro  F1 Macro  AUC-ROC
      Random Forest   0.7652           0.5514        0.5967    0.5532   0.6643
            XGBoost   0.8877           0.5808        0.5245    0.5262   0.6632
           LightGBM   0.8881           0.5796        0.5233    0.5243   0.6652
                SVM   0.6509           0.5498        0.6305    0.5135   0.6694
Logistic Regression   0.6251           0.5473        0.6277    0.4998   0.6785


## 5. Visualisasi, laporan XGBoost, dan penyimpanan model

Grafik membandingkan lima metrik. Confusion matrix dan classification report dicetak untuk XGBoost, yaitu model yang disimpan ke `models/xgboost_cross_sell_model.pkl`.

In [6]:
print("--> [INFO] Menyusun tabel komparasi, confusion matrix XGBoost, dan menyimpan model .pkl...")
plot_df = comparison.melt(id_vars="Model", var_name="Metrik", value_name="Nilai")
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.barplot(data=plot_df, x="Metrik", y="Nilai", hue="Model", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Perbandingan performa pada data uji")
ax.set_xlabel("")
ax.set_ylabel("Skor")
ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig(CHART_PATH, dpi=140, bbox_inches="tight")
plt.show()

xgb_model = fitted_models["XGBoost"]
xgb_pred = predictions["XGBoost"]
print("Classification report — XGBoost")
print(classification_report(y_test, xgb_pred, digits=4, target_names=["Tidak membeli (0)", "Membeli (1)"]))

cm = confusion_matrix(y_test, xgb_pred)
fig, ax = plt.subplots(figsize=(5.2, 4.4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Prediksi 0", "Prediksi 1"],
    yticklabels=["Aktual 0", "Aktual 1"],
    ax=ax,
)
ax.set_title("Confusion matrix — XGBoost")
fig.tight_layout()
fig.savefig(CM_PATH, dpi=140, bbox_inches="tight")
plt.show()
print("confusion matrix")
print(cm)

joblib.dump(xgb_model, MODEL_PATH)
print(f"Model tersimpan: {MODEL_PATH.name}")

<Figure size 1100x550 with 1 Axes>

Classification report — XGBoost
                   precision    recall  f1-score   support

Tidak membeli (0)     0.9058    0.9771    0.9401     51914
      Membeli (1)     0.2558    0.0719    0.1123      5686

         accuracy                         0.8877     57600
        macro avg     0.5808    0.5245    0.5262     57600
     weighted avg     0.8416    0.8877    0.8584     57600



<Figure size 520x440 with 1 Axes>

confusion matrix
[[50724  1190]
 [ 5277   409]]
Model tersimpan: xgboost_cross_sell_model.pkl
